# Unobserved Confounding Example

obs_df: DataFrame with columns 'treatment_obs' (T_i) and 'y_obs' (Y_i)
Model: Y_i = θ · T_i + z_i + ε_i where:
  - T_i ∈ {0, 1}
  - z_i ~ N(0, σ²) unobserved
 - Y_i observed with p(1/(1+e^(-z_i)))
  - ε_i ~ N(0, 0.5²)


In [ ]:
import os

os.environ['KERAS_BACKEND'] = 'jax'

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from tqdm import tqdm
from cmdstanpy import CmdStanModel

import keras
import bayesflow as bf

In [ ]:
# Parameters
np.random.seed(42)
n_population = 1000  # Full population size
n_samples = 100
param_names = ['treatment_effect', 'latent_confounder']

_w1 = np.random.normal(0, 1, size=1)
_b1 = np.random.normal(0, 1, size=1)
_w2 = np.random.normal(0, 1, size=1)
_b2 = np.random.normal(0, 1, size=1)
def selection_probability_black_box(z):
    """
    Black‐box selection probability:
      p = sigmoid( w2 * tanh(w1*z + b1) + b2 )
    This is nonlinear enough that no simple closed‐form inverse exists.
    """
    hidden = np.tanh(_w1 * z + _b1)
    logits = _w2 * hidden + _b2
    return 1 / (1 + np.exp(-logits))

def selection_probability_unknown(z):
    """beta distribution with unknown parameters"""
    p = np.random.uniform(0, 5, size=1)
    q = 1 / (1 + np.exp(-z))
    return np.random.beta(p, q)

def selection_probability(z):
    """Calculate selection probability p(1/(1+e^(-z)))"""
    return 1 / (1 + np.exp(-z))

def clinical_trial_model(treatment_effect: float = 1.7, latent_confounder: float = 1.):
    # Generate population
    z = latent_confounder * np.random.normal(0, 1, n_population)

    # generate treatment (e.g., if somebody has higher z, doctor things treatment is more effective, resulting in better outcome)
    prob_obs = selection_probability(z)
    #prob_obs = selection_probability_black_box(z)
    #prob_obs = selection_probability_unknown(z)
    treatment = np.random.binomial(1, 0.5, n_population)
    epsilon = np.random.normal(0, 0.5, n_population)
    y = treatment_effect * treatment + z + epsilon
    selected = np.random.binomial(1, prob_obs)

    # Ensure at least one observation
    if selected.sum() == 0:
        idx = np.random.choice(n_population)
        selected[idx] = 1

    # Subsample to desired sample size
    idx_obs = np.where(selected)[0]
    replace = len(idx_obs) < n_samples
    selected_index = np.random.choice(idx_obs, size=n_samples, replace=replace)
    
    # Create the observed sample
    observed_data = pd.DataFrame({
        'treatment_obs': treatment[selected_index],
        'y_obs': y[selected_index],
        'unobserved_confounder': z[selected_index]
    })
    
    # Full population (for reference)
    population_data = pd.DataFrame({
        'treatment_obs': treatment,
        'y_obs': y,
        'unobserved_confounder': z,
        'selected': selected
    })
    return observed_data, population_data

obs, pop = clinical_trial_model()
print(obs.shape)

In [ ]:
z = 0.5 * np.random.normal(0, 1, n_population)
prob_obs = selection_probability(z)
prob_obs_black = selection_probability_black_box(z)
prob_obs_black2 =selection_probability_unknown(z)

plt.hist(prob_obs, density=True, label='sigmoid')
plt.hist(prob_obs_black, density=True, label='black-box')
plt.hist(prob_obs_black2, density=True, label='unknown beta')
plt.title("Selection Probability Distribution")
plt.xlabel("Selection Probability")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
# Plot 1: Population vs Observed Distribution of Unobserved Confounder
fig1, ax1 = plt.subplots()
ax1.hist(pop["unobserved_confounder"], bins=30, alpha=0.6, label="Population", density=True)
ax1.hist(obs["unobserved_confounder"], bins=30, alpha=0.6, label="Selected (Observed)", density=True)
ax1.set_title("Distribution of Unobserved Confounder (z)")
ax1.set_xlabel("z (Health Awareness)")
ax1.set_ylabel("Density")
ax1.legend()

# Plot 2: Treatment vs Outcome in Observed Data
fig2, ax2 = plt.subplots()
ax2.boxplot([obs[obs["treatment_obs"] == 0]["y_obs"], obs[obs["treatment_obs"] == 1]["y_obs"]],
            tick_labels=["Control", "Treated"])
ax2.set_title("Observed Outcome by Treatment Group")
ax2.set_ylabel("y (Observed Outcome)")

# Plot 2: Treatment vs Outcome in Population Data
fig2, ax2 = plt.subplots()
ax2.boxplot([pop[pop["treatment_obs"] == 0]["y_obs"], pop[pop["treatment_obs"] == 1]["y_obs"]],
            tick_labels=["Control", "Treated"])
ax2.set_title("Observed Outcome by Treatment Group")
ax2.set_ylabel("y (Observed Outcome)")

# Show both plots
plt.tight_layout()
plt.show()

In [ ]:
# Function to compute naive treatment effect in observed PedCov
def estimate_treatment_effect(obs_df):
    treated_mean = obs_df[obs_df["treatment_obs"] == 1]["y_obs"].mean()
    control_mean = obs_df[obs_df["treatment_obs"] == 0]["y_obs"].mean()
    return treated_mean - control_mean

def estimate_treatment_effect_ols(obs_df):
    X = obs_df[["treatment_obs"]]
    y = obs_df["y_obs"]
    model = sm.OLS(y, X).fit(cov_type="HC3")
    return model.params["treatment_obs"]

def estimate_treatment_effect_ols_uncertainty(obs_df, draws):
    X = obs_df[["treatment_obs"]]
    #X = sm.add_constant(X)
    y = obs_df["y_obs"]
    model = sm.OLS(y, X).fit(cov_type="HC3")
    #model = sm.RLM(y, X, M=sm.robust.norms.AndrewWave()).fit()

    # Get point estimates and covariance matrix
    beta_hat = model.params["treatment_obs"]
    cov = model.cov_params()     # Var-cov matrix
    var = cov.loc["treatment_obs", "treatment_obs"]

    # Sample from approximate posterior
    posterior_samples_lr = np.random.normal(beta_hat, np.sqrt(var), size=draws)
    return posterior_samples_lr

In [ ]:
np.random.seed(42)
# Simulate multiple trials for each confounder strength to compute mean and standard error
n_trials = 100
confounder_strengths = np.linspace(0, 3.0, 20)
true_effect = 1.7
mean_effects = []
stderr_effects = []

for strength in confounder_strengths:
    estimates = []
    for _ in range(n_trials):
        obs_df, _ = clinical_trial_model(treatment_effect=true_effect, latent_confounder=strength)
        #est_effect = estimate_treatment_effect_ols(obs_df)
        est_effect = estimate_treatment_effect_ols_uncertainty(obs_df, draws=100)
        estimates.append(est_effect)
    estimates = np.array(estimates)
    mean_effects.append(estimates.mean())
    stderr_effects.append(estimates.std(ddof=1) / np.sqrt(n_trials))

# Plot: Estimated Treatment Effect with Standard Error Bands
fig, ax = plt.subplots()
ax.plot(confounder_strengths, mean_effects, label="Mean Estimated Effect", color="blue", marker='o')
ax.fill_between(confounder_strengths,
                np.array(mean_effects) - np.array(stderr_effects),
                np.array(mean_effects) + np.array(stderr_effects),
                color="blue", alpha=0.3, label="±1 SE")
ax.axhline(true_effect, color='red', linestyle='--', label="True Treatment Effect")
ax.set_xlabel("Strength of Unobserved Confounder (z coefficient)")
ax.set_ylabel("Estimated Treatment Effect")
ax.set_title("Bias and Uncertainty in Estimated Effect\nDue to Unobserved Confounding")
ax.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
prior_range = [0., 3.]
def single_simulation(latent_confounder=None):
    params = np.random.uniform(prior_range[0], prior_range[1], size=len(param_names))
    if not latent_confounder is None:  # we overwrite the samples
        params[1] = latent_confounder

    # Generate with clinical model with the given parameter
    df_i, _ = clinical_trial_model(params[0], latent_confounder=params[1])
    data_i = df_i[['y_obs', 'treatment_obs']].values
    return params, df_i, data_i

In [ ]:
def generative_model(batch_size, latent_confounder=None):
    # Initialize array
    data = np.zeros((batch_size, n_samples, 2))
    dfs = []
    param_batch = np.zeros((batch_size, len(param_names)))
    for i_p in range(batch_size):
        # Generate with clinical model with the given parameter
        params, df_i, data_i = single_simulation(latent_confounder=latent_confounder)
        data[i_p] = data_i
        param_batch[i_p] = params
        dfs.append(df_i)

    return {'sim_data': np.array(data), 'prior_draws': np.array(param_batch), 'obs_df': list(dfs)}

In [ ]:
np.random.seed(42)
valid_data = generative_model(100)

In [ ]:
estimates_OLS = []
for obs_df in valid_data['obs_df']:
    #est_effect = estimate_treatment_effect_ols(obs_df)
    est_effect = estimate_treatment_effect_ols_uncertainty(obs_df, draws=300)
    estimates_OLS.append(est_effect)
#estimates_OLS = np.array(estimates_OLS).reshape(len(valid_data['obs_df']), 1, 1)
estimates_OLS = np.array(estimates_OLS).reshape(len(valid_data['obs_df']), 300, 1)
print(bf.diagnostics.metrics.root_mean_squared_error(estimates_OLS, valid_data['prior_draws'][:, :1])['values'].mean())

bf.diagnostics.recovery(estimates_OLS, valid_data['prior_draws'][:, :1],
                        variable_names=['OLS ' + p for p in param_names[:1]]);
bf.diagnostics.calibration_ecdf(estimates_OLS, valid_data['prior_draws'][:, :1],
                                variable_names=['OLS ' + p for p in param_names[:1]],
                                difference=True);

In [ ]:
%%time

stan_model = CmdStanModel(stan_file='unobserved_stan_model.stan')
def get_stan_posterior(obs_df):

    # Prepare for Stan
    stan_data = {
        'N': obs_df.shape[0],
        'T': obs_df['treatment_obs'].values,
        'Y': obs_df['y_obs'].values,
    }

    # Fit the model to the
    fit = stan_model.sample(data=stan_data, show_progress=False)

    if len(param_names) == 1:
        return np.array(fit.draws_pd("theta"))
    return np.concatenate((fit.draws_pd("theta"), fit.draws_pd("sigma")), axis=-1)

estimates_STAN = []
for obs_df in tqdm(valid_data['obs_df']):
    est_effect = get_stan_posterior(obs_df)
    estimates_STAN.append(est_effect)
estimates_STAN = np.array(estimates_STAN).reshape(len(valid_data['obs_df']), -1, len(param_names))

stan_variable_names = ['STAN ' + p for p in param_names]
bf.diagnostics.recovery(estimates_STAN, valid_data['prior_draws'], variable_names=stan_variable_names)
bf.diagnostics.calibration_ecdf(estimates_STAN, valid_data['prior_draws'], difference=True,
                                variable_names=stan_variable_names);

# Use BayesFlow

In [ ]:
%%time
training_data = generative_model(10000)

In [ ]:
variable_names = ['Treatment Effect', 'Latent Confounder Variance']

In [ ]:
adapter = (
    bf.adapters.Adapter()
    .drop('obs_df')
    .to_array()
    .convert_dtype(from_dtype="float64", to_dtype="float32")
    .constrain("prior_draws", lower=0, upper=3, inclusive='both')
    .rename('prior_draws', to_key="inference_variables")
    .concatenate('sim_data', into="summary_variables")
)

In [ ]:
workflow = bf.BasicWorkflow(
    #simulator=generative_model,
    adapter=adapter,
    #summary_network=bf.networks.DeepSet(summary_dim=10),
    summary_network=bf.networks.SetTransformer(summary_dim=10),
    #inference_network=bf.networks.CouplingFlow(depth=5)
    inference_network=bf.networks.FlowMatching()
)

model_path = f'models/unobserved_variable_fm_ema.keras'
if os.path.exists(model_path):
    workflow.approximator = keras.saving.load_model(filepath=model_path)
else:
    history = workflow.fit_offline(
        data=training_data,
        epochs=5,
        batch_size=64,
        validation_data=valid_data,
    )
    #workflow.approximator.save(model_path)

In [ ]:
diagnostics = workflow.plot_default_diagnostics(test_data=valid_data, calibration_ecdf_kwargs={'difference': True},
                                                variable_names=variable_names)
#diagnostics['recovery'].savefig('plots/unobserved_recovery.pdf')

In [ ]:
# # check what happens when there is no bias
# test_data = generative_model(100, health_awareness=0.)
# diagnostics = workflow.plot_default_diagnostics(test_data=test_data, calibration_ecdf_kwargs={'difference': True},
#                                                 variable_names=variable_names)
# #diagnostics['recovery'].savefig('plots/unobserved_recovery_unbiased.pdf')

In [ ]:
posterior_samples = workflow.sample(num_samples=300, conditions=valid_data)

In [ ]:
bf_samples = posterior_samples['prior_draws'][:, :, :1]
stan_samples = estimates_STAN[:, -300:, :1]
#ols_samples = np.repeat(estimates_OLS, 300, axis=1)
ols_samples = estimates_OLS

estimates = np.concatenate((ols_samples, bf_samples, stan_samples), axis=-1)
target = np.stack((valid_data['prior_draws'][:, 0], valid_data['prior_draws'][:, 0],
                   valid_data['prior_draws'][:, 0]
                   ), axis=-1)
fig = bf.diagnostics.recovery(estimates, target, variable_names=['OLS', 'NPE', 'STAN'], figsize=(7, 2.5), add_corr=False)
#fig.savefig('plots/unobserved_variable_vs_stan.pdf')

estimates = np.concatenate((ols_samples, bf_samples), axis=-1)
target = np.stack((valid_data['prior_draws'][:, 0], valid_data['prior_draws'][:, 0]), axis=-1)
fig = bf.diagnostics.recovery(estimates, target, variable_names=['OLS', 'NPE'], figsize=(5, 2.5), add_corr=False)
#fig.savefig('plots/unobserved_variable_true.pdf')

estimates = np.concatenate((ols_samples, bf_samples), axis=-1)
target = np.stack((np.median(stan_samples, axis=1)[:, 0], np.median(stan_samples, axis=1)[:, 0]), axis=-1)
fig = bf.diagnostics.recovery(estimates, target, variable_names=['OLS', 'NPE'], xlabel='STAN Median', figsize=(5, 2.5), add_corr=False)
#fig.savefig('plots/unobserved_variable_stan.pdf')

In [ ]:
from scipy.stats import wasserstein_distance_nd

In [ ]:
wd = [wasserstein_distance_nd(estimates_STAN[i][-300:], posterior_samples['prior_draws'][i]) for i in range(100)]
wd